# 10年定着予測 - EDA v6由来のモデリング手続き改善（37_）

**背景**: `notebooks/eda/report/md/data_exploration_v6_report.md`（EDA v6）で、新規特徴量よりも
モデリング手続き側に大きな改善余地があることが実測で判明した。本ノートブックはその優先度1〜3を実装する。

特徴量は `28_`（現在の最良、L_v2_extended、Public 0.529454）と**完全に同一**で、1つも足していない。
変更点はモデルの学習・検証手続きのみ。

| # | 改善 | 根拠（v6レポート） | 期待幅 |
|---|---|---|---|
| 1 | 最終モデルを **Train全件** で再学習 | 第5節。現状80%しか学習に使っていない。学習曲線は80%地点でも未飽和 | 0.005〜0.01 |
| 2 | **シード平均**（5シード）で提出 | 第6節。単一シードのsdは0.0045〜0.0068で、判定してきた効果量より大きい | 0.006〜0.009 |
| 3 | 検証セットから**早期退職者を除外** | 第1〜2節。Testには0-23ヶ月の退職者が0名。検証-Publicギャップ平均が0.0184→0.0070 | 水準の補正 |

**注意（3について）**: 早期退職者は**検証セットからのみ除外**し、学習からは除かない。
v6の追試で学習からも除くと僅かに悪化した（0.5326→0.5370）ため。

## 実行構成

| config | 学習データ | 検証セット | シード | 位置づけ |
|---|---|---|---|---|
| `A_repro_28` | 先頭80% | 全体（= `28_`と同一） | 1 | **再現性チェック**。val≈0.503065になるはず |
| `B_valfix` | 先頭80% | 生存者のみ | 1 | 改善3のみ |
| `C_valfix_seedavg` | 先頭80% | 生存者のみ | 5 | 改善3+2 |
| `D_full_seedavg_x125` | **全件** | なし | 5 | 改善3+2+1 ← **本命** |
| `D2_full_seedavg_x100` | **全件** | なし | 5 | 反復数スケールを変えた保険 |

`D`系は学習に全データを使うため**検証スコアが原理的に計算できない**。Publicでしか評価できない点に注意。

## 実行環境

Google Colab Pro の **CPUハイメモリ**ランタイムを想定（`18_`〜`35_`と同様、GPUランタイムは不要）。
想定実行時間は Optuna 2スタディ（各25試行）＋最終学習17回で **約1〜2時間**。

> ⚠️ **ローカルMacで先行実行しないこと。** `27_`で発生したチェックポイントのGoogle Drive同期事故
> （ローカルで作ったチェックポイントをColabが「計算済み」と誤認する）を避けるため、本ノートブックは
> Colabで直接実行する。やり直したい場合は下の `RESET_CHECKPOINT = True` にする。

In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 28.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 16.9 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "37_full_train_seed_averaging"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-11 14:24:55] [INFO] === [37_full_train_seed_averaging] 実験開始 ===


INFO:37_full_train_seed_averaging:=== [37_full_train_seed_averaging] 実験開始 ===


[2026-08-11 14:24:56] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


INFO:37_full_train_seed_averaging:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


[2026-08-11 14:24:56] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/37_full_train_seed_averaging_checkpoint.csv


INFO:37_full_train_seed_averaging:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/37_full_train_seed_averaging_checkpoint.csv


[2026-08-11 14:24:56] [INFO] チェックポイントは未作成（新規実行）


INFO:37_full_train_seed_averaging:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-11 14:25:01] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:37_full_train_seed_averaging:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-11 14:25:01] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:37_full_train_seed_averaging:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-11 14:25:01] [INFO] 定着率: 0.5647


INFO:37_full_train_seed_averaging:定着率: 0.5647


[2026-08-11 14:25:01] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:37_full_train_seed_averaging:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-11 14:25:01] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:37_full_train_seed_averaging:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-11 14:25:01] [INFO] Test  早期退職者: 0名 / 2502名


INFO:37_full_train_seed_averaging:Test  早期退職者: 0名 / 2502名


[2026-08-11 14:25:01] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:37_full_train_seed_averaging:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-11 14:25:01] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:37_full_train_seed_averaging:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-11 14:25:02] [INFO] ------------------------------------------------------------


INFO:37_full_train_seed_averaging:------------------------------------------------------------


[2026-08-11 14:25:02] [INFO] split非依存の基本特徴量を生成中...


INFO:37_full_train_seed_averaging:split非依存の基本特徴量を生成中...


[2026-08-11 14:25:02] [INFO] ------------------------------------------------------------


INFO:37_full_train_seed_averaging:------------------------------------------------------------


[2026-08-11 14:30:37] [INFO] split非依存の基本特徴量生成完了


INFO:37_full_train_seed_averaging:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-11 14:30:37] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:37_full_train_seed_averaging:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-11 14:30:38] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:37_full_train_seed_averaging:入社時メモ: SVD累積寄与率=0.760


[2026-08-11 14:30:42] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:37_full_train_seed_averaging:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-11 14:30:44] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:37_full_train_seed_averaging:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-11 14:30:44] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:37_full_train_seed_averaging:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-11 14:30:44] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:37_full_train_seed_averaging:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-11 14:32:42] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:37_full_train_seed_averaging:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-11 14:32:42] [INFO] Persona単位の基本特徴量を生成中...


INFO:37_full_train_seed_averaging:Persona単位の基本特徴量を生成中...


[2026-08-11 14:32:42] [INFO] Persona単位の基本特徴量処理完了


INFO:37_full_train_seed_averaging:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-11 14:32:43] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:37_full_train_seed_averaging:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-11 14:32:43] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:37_full_train_seed_averaging:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-11 14:32:43] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:37_full_train_seed_averaging:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`extra_blocks`パラメータで`{"L1"}`/`{"L2"}`を指定し、ベースライン
（D_expanded + TF-IDF A_v1、`18_`の構成、Eなし）に対してL_v1・L_v2のいずれかを単体で追加できるようにする。

In [14]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）")

✅ 部署Target Encoding・prepare_split関数定義完了（37_版: 全件学習・検証セット補正に対応）


## 7. チェックポイント機能（`18_`〜`28_`をベースに、37_で固定スキーマ化）

`28_`までは全configが同じキーを持っていたが、37_ は A / BC / D で記録すべき情報が異なる。
キー構成がバラバラのまま `mode="a"` でCSVに追記すると列がずれて壊れるため、
`RESULT_SCHEMA` に揃えてから書き出す。

In [15]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


## 8. モデル関数（37_版）

`28_`の `run_model_config` を3つに分解する。

- `tune_hyperparams`: Optunaで探索（探索空間は`18_`〜`28_`と完全に同一、n_trials=25）
- `fit_holdout`: 80/20で学習し、early stoppingで最良反復数を決める。シードを変えて複数回実行できる
- `fit_full_train`: **Train全件**で学習する（検証セットが無いので反復数は固定、early stoppingなし）

シード平均は、同一パラメータ・同一特徴量のままシードだけ変えたモデルの**予測確率を単純平均**する。
重みを一切学習しないので、`11_`/`12_`/`32_`で失敗した「OOFから重みを学習するアンサンブル」とは
別物であり、過去の教訓には抵触しない。

In [16]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 9. 特徴量の組み立て

ブロックは `L2`（= `28_`の `L_v2_extended`、現在の最良）に固定する。

- `split_80_20` × 検証=全体 → config A（`28_`の完全再現）
- `split_80_20` × 検証=生存者のみ → config B / C
- `split_100`（全件） → config D / D2

In [17]:
BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[A用] split_80_20 / 検証=全体（28_と同一）")
ag_train_80, ag_val_all, test_features = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=False)

logger.info("[B,C用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[D用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"A: train={len(ag_train_80)}, val={len(ag_val_all)}（早期退職者を含む）")
logger.info(f"B/C: train={len(ag_train_80b)}, val={len(ag_val_surv)}（生存者のみ）")
logger.info(f"D: train={len(ag_full)}（全件）, val={len(ag_empty)}（空）")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80))}")

# 生存者マスク（Aの検証予測を生存者だけで採点し直すのに使う）
SURV_MASK_A = ~ag_val_all.index.isin(EARLY_LEAVER_IDS)
assert len(ag_train_80) == len(ag_train_80b), "A と B/C の学習データは同一のはず"
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"

[2026-08-11 14:32:44] [INFO] ============================================================


INFO:37_full_train_seed_averaging:============================================================


[2026-08-11 14:32:44] [INFO] [A用] split_80_20 / 検証=全体（28_と同一）


INFO:37_full_train_seed_averaging:[A用] split_80_20 / 検証=全体（28_と同一）


[2026-08-11 14:32:44] [INFO] [B,C用] split_80_20 / 検証=生存者のみ


...

INFO:37_full_train_seed_averaging:B/C: train=2208, val=535（生存者のみ）


[2026-08-11 14:32:44] [INFO] D: train=2761（全件）, val=0（空）


INFO:37_full_train_seed_averaging:D: train=2761（全件）, val=0（空）


[2026-08-11 14:32:44] [INFO] 特徴量数: 441


INFO:37_full_train_seed_averaging:特徴量数: 441


## 10. config A: `28_`の再現性チェック

`28_`の `L_v2_extended`（split_80_20）と完全に同じ設定。Colab CPUでは決定論的なので、
**val_score が 0.503065 に一致すれば**特徴量パイプラインが `28_` と同一であることが確認できる
（`27_`・`28_`・`34_`・`35_`でも同じチェックを行っている）。

同時に、この検証予測を**生存者だけで採点し直した値**も出す。これが config B/C/D と比較すべき数値になる。

In [18]:
def run_config_A():
    logger.info("=" * 60); logger.info("[A_repro_28] Optuna探索（検証=全体）")
    params_A = tune_hyperparams(ag_train_80, ag_val_all)
    out = fit_holdout(ag_train_80, ag_val_all, test_features, params_A, seeds=[SEED])
    val_all = log_loss(out["y_val"], out["val_preds"][0])
    val_surv = log_loss(out["y_val"][SURV_MASK_A], out["val_preds"][0][SURV_MASK_A])
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_A_repro_28_valpreds.npy", out["val_preds"][0])
    path = save_submission(test_features.index, out["test_preds"][0], "A_repro_28")
    logger.info(f"[A_repro_28] val(全体)={val_all:.6f}  ← 28_の実測値 0.503065 と比較")
    logger.info(f"[A_repro_28] val(生存者のみ)={val_surv:.6f}  ← B/C/Dと比較すべき数値")
    return make_row(config="A_repro_28", n_features=len(out["feature_cols"]),
                    val_score=val_surv, val_score_all=val_all,
                    best_iter=out["best_iters"][0], params=json.dumps(params_A),
                    submission_path=path)

import json
result_A = run_or_resume("A_repro_28", run_config_A)
print(f"val(全体) = {result_A['val_score_all']:.6f}   （28_実測値: 0.503065、差 {abs(float(result_A['val_score_all']) - 0.503065):.6f}）")
print(f"val(生存者のみ) = {result_A['val_score']:.6f}")

[2026-08-11 14:32:44] [INFO] ============================================================


INFO:37_full_train_seed_averaging:============================================================


[2026-08-11 14:32:44] [INFO] [A_repro_28] Optuna探索（検証=全体）


INFO:37_full_train_seed_averaging:[A_repro_28] Optuna探索（検証=全体）


[2026-08-11 14:35:17] [INFO]   Optuna完了: best_value=0.503713, best_params={'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


...

[2026-08-11 14:35:23] [INFO] [A_repro_28] val(全体)=0.503065  ← 28_の実測値 0.503065 と比較


INFO:37_full_train_seed_averaging:[A_repro_28] val(全体)=0.503065  ← 28_の実測値 0.503065 と比較


[2026-08-11 14:35:23] [INFO] [A_repro_28] val(生存者のみ)=0.514254  ← B/C/Dと比較すべき数値


INFO:37_full_train_seed_averaging:[A_repro_28] val(生存者のみ)=0.514254  ← B/C/Dと比較すべき数値


val(全体) = 0.503065   （28_実測値: 0.503065、差 0.000000）
val(生存者のみ) = 0.514254


## 11. config B / C: 検証セット補正（改善3）＋ シード平均（改善2）

検証セットから早期退職者18名を除いた状態でOptunaを回し直し、その最良パラメータで
シードを変えて5回学習する。

- **B** = 5シードのうち先頭1つ（改善3のみの効果を見る）
- **C** = 5シードの予測確率を平均（改善3+2）

Optunaは1回だけ回し、B/Cで共有する。

In [19]:
def run_config_BC():
    logger.info("=" * 60); logger.info("[B/C] Optuna探索（検証=生存者のみ）")
    params_BC = tune_hyperparams(ag_train_80b, ag_val_surv)
    out = fit_holdout(ag_train_80b, ag_val_surv, test_features, params_BC, seeds=SEEDS)

    single_scores = [log_loss(out["y_val"], vp) for vp in out["val_preds"]]
    avg_val_pred = out["val_preds"].mean(axis=0)
    val_seedavg = log_loss(out["y_val"], avg_val_pred)

    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_C_valfix_seedavg_valpreds.npy", avg_val_pred)
    path_B = save_submission(test_features.index, out["test_preds"][0], "B_valfix")
    path_C = save_submission(test_features.index, out["test_preds"].mean(axis=0), "C_valfix_seedavg")

    logger.info(f"[B] 単一シード val: {single_scores[0]:.6f}")
    logger.info(f"[B] 5シードの val: mean={np.mean(single_scores):.6f}, sd={np.std(single_scores):.6f}, "
                f"min={np.min(single_scores):.6f}, max={np.max(single_scores):.6f}")
    logger.info(f"[C] シード平均 val: {val_seedavg:.6f}（単一シード平均比 {val_seedavg - np.mean(single_scores):+.6f}）")
    logger.info(f"[B/C] best_iteration: {out['best_iters']} → 平均 {np.mean(out['best_iters']):.0f}")

    return make_row(config="BC_valfix", n_features=len(out["feature_cols"]),
                    val_score=val_seedavg, val_score_single=single_scores[0],
                    val_single_mean=float(np.mean(single_scores)),
                    val_single_sd=float(np.std(single_scores)),
                    best_iter=float(np.mean(out["best_iters"])), params=json.dumps(params_BC),
                    submission_path=f"{path_B}|{path_C}")

result_BC = run_or_resume("BC_valfix", run_config_BC)
print(f"B 単一シード val      = {result_BC['val_score_single']:.6f}")
print(f"  5シードの val sd    = {result_BC['val_single_sd']:.6f}  ← v6で測った0.0045〜0.0068と同水準か確認")
print(f"C シード平均 val      = {result_BC['val_score']:.6f}")
print(f"A 生存者のみ val      = {float(result_A['val_score']):.6f}  ← 比較対象")

[2026-08-11 14:35:23] [INFO] ============================================================


INFO:37_full_train_seed_averaging:============================================================


[2026-08-11 14:35:23] [INFO] [B/C] Optuna探索（検証=生存者のみ）


INFO:37_full_train_seed_averaging:[B/C] Optuna探索（検証=生存者のみ）


[2026-08-11 14:40:23] [INFO]   Optuna完了: best_value=0.519838, best_params={'depth': 8, 'learning_rate': 0.02490240331753947, 'l2_leaf_reg': 0.08626914949556161, 'border_count': 44, 'bagging_temperature': 0.6092599562105125, 'random_strength': 0.30378392360897394}


...

[2026-08-11 14:40:51] [INFO] [C] シード平均 val: 0.521978（単一シード平均比 -0.008297）


INFO:37_full_train_seed_averaging:[C] シード平均 val: 0.521978（単一シード平均比 -0.008297）


[2026-08-11 14:40:51] [INFO] [B/C] best_iteration: [242, 88, 147, 153, 195] → 平均 165


INFO:37_full_train_seed_averaging:[B/C] best_iteration: [242, 88, 147, 153, 195] → 平均 165


B 単一シード val      = 0.519838
  5シードの val sd    = 0.007776  ← v6で測った0.0045〜0.0068と同水準か確認
C シード平均 val      = 0.521978
A 生存者のみ val      = 0.514254  ← 比較対象


## 12. config D / D2: Train全件で再学習（改善1）＋ シード平均

`28_`は最終モデルを Train の80%（2,208名）でしか学習しておらず、残り553名は early stopping 用の
検証にしか使っていなかった。ここでは**全件（2,761名）**で学習する。

検証セットが無くなるため early stopping が使えない。反復数は config B/C で得られた
`best_iteration` の平均を、データ件数比（2761/2208 ≒ 1.25）でスケールして固定する。
このスケール係数だけは理論的な根拠が無い任意の選択なので、**1.25倍と1.00倍の両方**を作って
保険とする。

> **重要**: D・D2 は学習に全データを使うため、**検証スコアが原理的に計算できない**。
> Public でしか評価できない。

In [20]:
def make_run_config_D(tag, scale):
    def _run():
        best_params = json.loads(result_BC["params"])
        base_iter = float(result_BC["best_iter"])
        n_iter = max(int(base_iter * scale), 50)
        logger.info("=" * 60)
        logger.info(f"[D_full_seedavg_{tag}] Train全件学習: iterations={n_iter} "
                    f"(= B/Cのbest_iteration平均 {base_iter:.0f} × {scale})")
        preds = fit_full_train(ag_full, test_features_full, best_params, n_iter, seeds=SEEDS)
        path = save_submission(test_features_full.index, preds.mean(axis=0), f"D_full_seedavg_{tag}")
        return make_row(config=f"D_full_seedavg_{tag}", n_features=len(_feature_cols(ag_full)),
                        n_iterations=n_iter, params=result_BC["params"], submission_path=path)
    return _run

result_D  = run_or_resume("D_full_seedavg_x125", make_run_config_D("x125", ITER_SCALE_CANDIDATES["x125"]))
result_D2 = run_or_resume("D_full_seedavg_x100", make_run_config_D("x100", ITER_SCALE_CANDIDATES["x100"]))
print("D  (x1.25):", result_D["submission_path"])
print("D2 (x1.00):", result_D2["submission_path"])

[2026-08-11 14:40:52] [INFO] ============================================================


INFO:37_full_train_seed_averaging:============================================================


[2026-08-11 14:40:52] [INFO] [D_full_seedavg_x125] Train全件学習: iterations=206 (= B/Cのbest_iteration平均 165 × 1.25)


INFO:37_full_train_seed_averaging:[D_full_seedavg_x125] Train全件学習: iterations=206 (= B/Cのbest_iteration平均 165 × 1.25)


[2026-08-11 14:40:56] [INFO]   seed=42: 全件学習完了（iterations=206）


...

[2026-08-11 14:41:29] [INFO]   seed=99: 全件学習完了（iterations=165）


INFO:37_full_train_seed_averaging:  seed=99: 全件学習完了（iterations=165）


[2026-08-11 14:41:29] [INFO]   提出ファイル: 20260811_37_full_train_seed_averaging_D_full_seedavg_x100.csv（予測平均=0.5951）


INFO:37_full_train_seed_averaging:  提出ファイル: 20260811_37_full_train_seed_averaging_D_full_seedavg_x100.csv（予測平均=0.5951）


D  (x1.25): /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_37_full_train_seed_averaging_D_full_seedavg_x125.csv
D2 (x1.00): /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_37_full_train_seed_averaging_D_full_seedavg_x100.csv


## 13. 総合結果・提出候補

In [21]:
rows = [
    {"config": "A_repro_28（= 28_の再現）", "学習": "先頭80%", "検証": "全体", "シード": 1,
     "val(生存者のみ)": float(result_A["val_score"]), "val(全体)": float(result_A["val_score_all"]),
     "submission": Path(result_A["submission_path"]).name},
    {"config": "B_valfix（改善3）", "学習": "先頭80%", "検証": "生存者のみ", "シード": 1,
     "val(生存者のみ)": float(result_BC["val_score_single"]), "val(全体)": np.nan,
     "submission": Path(result_BC["submission_path"].split("|")[0]).name},
    {"config": "C_valfix_seedavg（改善3+2）", "学習": "先頭80%", "検証": "生存者のみ", "シード": len(SEEDS),
     "val(生存者のみ)": float(result_BC["val_score"]), "val(全体)": np.nan,
     "submission": Path(result_BC["submission_path"].split("|")[1]).name},
    {"config": "D_full_seedavg_x125（改善3+2+1）★本命", "学習": "全件", "検証": "—", "シード": len(SEEDS),
     "val(生存者のみ)": np.nan, "val(全体)": np.nan, "submission": Path(result_D["submission_path"]).name},
    {"config": "D2_full_seedavg_x100（保険）", "学習": "全件", "検証": "—", "シード": len(SEEDS),
     "val(生存者のみ)": np.nan, "val(全体)": np.nan, "submission": Path(result_D2["submission_path"]).name},
]
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv", index=False)

logger.info("=" * 60)
logger.info("[サマリ]")
for r in rows:
    logger.info(f"  {r['config']}: val(生存者)={r['val(生存者のみ)']}, file={r['submission']}")

print("\n【判定の目安】")
print(f"1. A の val(全体) が 0.503065 に一致 → 特徴量パイプラインは 28_ と同一（差: "
      f"{abs(float(result_A['val_score_all']) - 0.503065):.6f}）")
print(f"2. C(シード平均) < B(単一シード) なら、改善2が検証上も効いている "
      f"（差: {float(result_BC['val_score']) - float(result_BC['val_score_single']):+.6f}）")
print("3. D は検証不能。EDA v6 第5節の学習曲線（80%地点で未飽和）を根拠に提出する")

,config,学習,検証,シード,val(生存者のみ),val(全体),submission
0,A_repro_28（= 28_の再現）,先頭80%,全体,1,0.514254,0.503065,20260811_37_full_train_seed_averaging_A_repro_...
1,B_valfix（改善3）,先頭80%,生存者のみ,1,0.519838,NaN,20260811_37_full_train_seed_averaging_B_valfix...
2,C_valfix_seedavg（改善3+2）,先頭80%,生存者のみ,5,0.521978,NaN,20260811_37_full_train_seed_averaging_C_valfix...
3,D_full_seedavg_x125（改善3+2+1）★本命,全件,—,5,NaN,NaN,20260811_37_full_train_seed_averaging_D_full_s...
4,D2_full_seedavg_x100（保険）,全件,—,5,NaN,NaN,20260811_37_full_train_seed_averaging_D_full_s...


[2026-08-11 14:41:29] [INFO] ============================================================


INFO:37_full_train_seed_averaging:============================================================


[2026-08-11 14:41:29] [INFO] [サマリ]


INFO:37_full_train_seed_averaging:[サマリ]


...

[2026-08-11 14:41:29] [INFO]   D_full_seedavg_x125（改善3+2+1）★本命: val(生存者)=nan, file=20260811_37_full_train_seed_averaging_D_full_seedavg_x125.csv


INFO:37_full_train_seed_averaging:  D_full_seedavg_x125（改善3+2+1）★本命: val(生存者)=nan, file=20260811_37_full_train_seed_averaging_D_full_seedavg_x125.csv


[2026-08-11 14:41:29] [INFO]   D2_full_seedavg_x100（保険）: val(生存者)=nan, file=20260811_37_full_train_seed_averaging_D_full_seedavg_x100.csv


INFO:37_full_train_seed_averaging:  D2_full_seedavg_x100（保険）: val(生存者)=nan, file=20260811_37_full_train_seed_averaging_D_full_seedavg_x100.csv



【判定の目安】
1. A の val(全体) が 0.503065 に一致 → 特徴量パイプラインは 28_ と同一（差: 0.000000）
2. C(シード平均) < B(単一シード) なら、改善2が検証上も効いている （差: +0.002139）
3. D は検証不能。EDA v6 第5節の学習曲線（80%地点で未飽和）を根拠に提出する


## 14. 提出方針と注意

### 提出の優先順位

1. **`D_full_seedavg_x125`（本命）** — 改善1+2+3を全部入れた構成。EDA v6の実測から
   0.01〜0.02程度の改善を期待するが、**全件学習は検証で確認できない**ため、これが最初の実地検証になる。
2. **`C_valfix_seedavg`** — Dが期待外れだった場合の切り分け用。全件学習を除いた（＝検証可能な範囲の）
   改善だけを含むので、「シード平均は効いたが全件学習が悪かった」のか
   「そもそも両方効かなかった」のかを分離できる。
3. `D2_full_seedavg_x100` — Dが悪化した場合に、反復数スケールが原因かどうかを見る保険。

`A_repro_28` は再現性チェック用なので提出不要（`28_`と同じPublic 0.529454になるはず）。

### 結果の解釈で注意すべき点

- **Dが悪化したら、その原因は「全件学習」か「反復数の決め方」のどちらか**。
  全件学習自体は理論的に有利なので、まず `D2`（スケール1.00）で反復数の影響を切り分ける。
- 期待幅（0.005〜0.01 + 0.006〜0.009）は**軽量な特徴量セットでの実測値**であり、
  本番の441列パイプラインでそのまま再現する保証はない。EDA v6 第5節の学習曲線には
  70%地点の非単調性（0.54649 > 60%の0.54470）もあった。
- 改善3（検証セット補正）は**Publicスコアを直接動かす改善ではない**。検証スコアの水準を
  Publicと同じ土俵に載せるための修正であり、その効果はconfig B と A の val(生存者のみ) の
  比較として現れる（Optunaの探索対象が変わるため、間接的にはPublicにも影響しうる）。

### 次のアクション

- Public結果を `data/output/submit_result_report.md` に追記する。
- EDA v6 の優先度4（生存時間情報の活用: ソフトラベル / h84補助目的変数）は別途検証中。
  こちらが有望なら `38_` として実装する。

---

## 15. 追加検証: Aのハイパーパラメータで改善1+2をやり直す（1回目の実行結果を受けて追加）

### 1回目の実行で分かったこと

| config | Optunaの目的 | 選ばれたパラメータ | best_iter | val(生存者のみ) |
|---|---|---|---|---|
| A_repro_28 | val 全体 | depth=**4**, lr=0.0352, **l2=2.22**, border=218 | **584** | **0.51425** |
| B_valfix（seed42） | val 生存者のみ | depth=**8**, lr=0.0249, **l2=0.086**, border=44 | **165** | 0.51984 |
| C_valfix_seedavg（5シード平均） | 同上 | 同上 | 165 | 0.52198 |

- ✅ **改善2（シード平均）は予想通り効いた**: 単一シード平均 0.53027 → 5シード平均 0.52198（**-0.0083**）。
  EDA v6 の予測（0.006〜0.009）とほぼ一致。シードsdも **0.00778** とv6の実測（0.0045〜0.0068）より更に大きく、
  「単一実行同士の比較でブロック採否を判定してはいけない」ことが本番441列でも確認された。
- ⚠️ **改善3（検証セット補正）は裏目に出た可能性がある**: 検証セットが18名（3.3%）変わっただけで
  Optunaが**まったく別の領域**に着地した（depth 4→8、L2正則化が26分の1）。
  best_iterationが584→165に落ちているのは典型的な過学習のサインで、深い木＋ほぼ無正則化という
  組み合わせが選ばれてしまっている。

  ただし A(0.51425) と B(0.51984) の差 0.0056 は**シードsd(0.0078)より小さい**ため、
  「改善3が悪い」と結論するだけの証拠は無い。言えるのは「改善3が効いたという証拠も無く、
  ハイパーパラメータが大きくブレた」ということだけ。

### なぜ D をこのまま提出しない方がよいか

**D / D2 は B/C のハイパーパラメータ（depth=8, l2=0.086）を引き継いでいる**。
つまり D が期待外れだった場合、原因が「全件学習が悪かった」のか「ハイパーパラメータが悪かった」のか
**切り分けられない**。反復数 206 も、その過学習気味な設定から得た best_iter=165 に由来している。

### 追加する構成

`A_repro_28` のパラメータは **Public 0.529454 を実際に出した既知の良い設定**である
（A の提出ファイルは `28_` と完全に一致することを確認済み）。この既知の良い設定の上に、
**証拠のある改善（1と2）だけ**を乗せる。

| config | 学習 | パラメータ | シード | 位置づけ |
|---|---|---|---|---|
| `C2_Aparams_seedavg` | 先頭80% | **Aの設定** | 5 | Aの設定でのシード平均効果を測る |
| `D3_Aparams_full_x125` | **全件** | **Aの設定** | 5 | **改善1+2のみ。本命を差し替え** |

Optunaは再探索しない（Aの結果を再利用）ので、追加コストは学習10回分だけ（数分）。

In [22]:
def run_config_C2():
    params_A = json.loads(result_A["params"])
    logger.info("=" * 60)
    logger.info(f"[C2_Aparams_seedavg] Aのパラメータ + シード平均: {params_A}")
    out = fit_holdout(ag_train_80b, ag_val_surv, test_features, params_A, seeds=SEEDS)

    single_scores = [log_loss(out["y_val"], vp) for vp in out["val_preds"]]
    avg_val_pred = out["val_preds"].mean(axis=0)
    val_seedavg = log_loss(out["y_val"], avg_val_pred)

    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_C2_Aparams_seedavg_valpreds.npy", avg_val_pred)
    path = save_submission(test_features.index, out["test_preds"].mean(axis=0), "C2_Aparams_seedavg")

    logger.info(f"[C2] 5シードの val: mean={np.mean(single_scores):.6f}, sd={np.std(single_scores):.6f}")
    logger.info(f"[C2] シード平均 val: {val_seedavg:.6f}")
    logger.info(f"[C2] best_iteration: {out['best_iters']} → 平均 {np.mean(out['best_iters']):.0f}")

    return make_row(config="C2_Aparams_seedavg", n_features=len(out["feature_cols"]),
                    val_score=val_seedavg, val_score_single=single_scores[0],
                    val_single_mean=float(np.mean(single_scores)),
                    val_single_sd=float(np.std(single_scores)),
                    best_iter=float(np.mean(out["best_iters"])), params=json.dumps(params_A),
                    submission_path=path)

result_C2 = run_or_resume("C2_Aparams_seedavg", run_config_C2)

print("=== val(生存者のみ) 比較 ===")
print(f"  A  （Aの設定, 1シード, 80%学習） : {float(result_A['val_score']):.6f}")
print(f"  B  （BCの設定, 1シード, 80%学習）: {float(result_BC['val_score_single']):.6f}")
print(f"  C  （BCの設定, 5シード, 80%学習）: {float(result_BC['val_score']):.6f}")
print(f"  C2 （Aの設定, 5シード, 80%学習） : {float(result_C2['val_score']):.6f}  ← 最良になるはず")
print(f"\n  C2の単一シード平均 {float(result_C2['val_single_mean']):.6f} → シード平均 {float(result_C2['val_score']):.6f} "
      f"（{float(result_C2['val_score']) - float(result_C2['val_single_mean']):+.6f}）")

[2026-08-11 14:49:48] [INFO] ============================================================


INFO:37_full_train_seed_averaging:============================================================


[2026-08-11 14:49:48] [INFO] [C2_Aparams_seedavg] Aのパラメータ + シード平均: {'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


INFO:37_full_train_seed_averaging:[C2_Aparams_seedavg] Aのパラメータ + シード平均: {'depth': 4, 'learning_rate': 0.03518359458951149, 'l2_leaf_reg': 2.217690447016724, 'border_count': 218, 'bagging_temperature': 0.6787467566574921, 'random_strength': 1.438494697238285}


[2026-08-11 14:49:55] [INFO]   seed=42: val_logloss=0.514254, best_iteration=584


...

[2026-08-11 14:50:13] [INFO] [C2] シード平均 val: 0.517698


INFO:37_full_train_seed_averaging:[C2] シード平均 val: 0.517698


[2026-08-11 14:50:13] [INFO] [C2] best_iteration: [584, 506, 382, 405, 366] → 平均 449


INFO:37_full_train_seed_averaging:[C2] best_iteration: [584, 506, 382, 405, 366] → 平均 449


=== val(生存者のみ) 比較 ===
  A  （Aの設定, 1シード, 80%学習） : 0.514254
  B  （BCの設定, 1シード, 80%学習）: 0.519838
  C  （BCの設定, 5シード, 80%学習）: 0.521978
  C2 （Aの設定, 5シード, 80%学習） : 0.517698  ← 最良になるはず

  C2の単一シード平均 0.520469 → シード平均 0.517698 （-0.002771）


In [23]:
def run_config_D3():
    params_A = json.loads(result_A["params"])
    base_iter = float(result_C2["best_iter"])
    n_iter = max(int(base_iter * 1.25), 50)
    logger.info("=" * 60)
    logger.info(f"[D3_Aparams_full_x125] Aの設定でTrain全件学習: iterations={n_iter} "
                f"(= C2のbest_iteration平均 {base_iter:.0f} × 1.25)")
    preds = fit_full_train(ag_full, test_features_full, params_A, n_iter, seeds=SEEDS)
    path = save_submission(test_features_full.index, preds.mean(axis=0), "D3_Aparams_full_x125")
    return make_row(config="D3_Aparams_full_x125", n_features=len(_feature_cols(ag_full)),
                    n_iterations=n_iter, params=json.dumps(params_A), submission_path=path)

result_D3 = run_or_resume("D3_Aparams_full_x125", run_config_D3)
print("D3 (本命):", result_D3["submission_path"])

[2026-08-11 14:50:22] [INFO] ============================================================


INFO:37_full_train_seed_averaging:============================================================


[2026-08-11 14:50:22] [INFO] [D3_Aparams_full_x125] Aの設定でTrain全件学習: iterations=560 (= C2のbest_iteration平均 449 × 1.25)


INFO:37_full_train_seed_averaging:[D3_Aparams_full_x125] Aの設定でTrain全件学習: iterations=560 (= C2のbest_iteration平均 449 × 1.25)


[2026-08-11 14:50:27] [INFO]   seed=42: 全件学習完了（iterations=560）


...

[2026-08-11 14:50:49] [INFO]   seed=99: 全件学習完了（iterations=560）


INFO:37_full_train_seed_averaging:  seed=99: 全件学習完了（iterations=560）


[2026-08-11 14:50:49] [INFO]   提出ファイル: 20260811_37_full_train_seed_averaging_D3_Aparams_full_x125.csv（予測平均=0.5875）


INFO:37_full_train_seed_averaging:  提出ファイル: 20260811_37_full_train_seed_averaging_D3_Aparams_full_x125.csv（予測平均=0.5875）


D3 (本命): /content/drive/MyDrive/jaggle_2026/data/output/20260811/20260811_37_full_train_seed_averaging_D3_Aparams_full_x125.csv


## 16. 最終的な提出方針（1回目の結果を反映して更新）

### 提出順

1. **`D3_Aparams_full_x125`（本命）** — Public 0.529454 を出した既知の良いハイパーパラメータに、
   証拠のある改善（全件学習＋シード平均）だけを乗せた構成。
2. **`C2_Aparams_seedavg`** — D3が期待外れだった場合の切り分け用。全件学習を除いた
   （＝検証可能な範囲の）改善だけを含むので、「シード平均は効いたが全件学習が悪かった」のか
   「両方効かなかった」のかを分離できる。
3. `D_full_seedavg_x125` / `C_valfix_seedavg` — 改善3（検証セット補正）由来の
   ハイパーパラメータを使う系統。上の2つが両方ダメだった場合にのみ検討する。

### 提出しないもの

- **`A_repro_28`** — `28_` の提出ファイルと**完全に一致**（相関1.0000、最大差0.0000）することを
  確認済み。提出しても Public 0.529454 が返ってくるだけなので、提出回数の無駄。

### 確定した知見（Public確認前でも言えること）

- **特徴量パイプラインは `28_` と完全に同一**: A の val(全体)=0.5030654 が `28_` の実測値 0.503065 と
  小数点以下6桁まで一致し、提出ファイルも完全一致した。
- **改善2（シード平均）は本番441列でも実測で効く**: 単一シード平均 0.53027 → 5シード平均 0.52198。
  シードsdは 0.00778 で、これまで採否を判定してきた効果量（L_v1→L_v2 の Public差 0.0002 など）より
  はるかに大きい。**今後のアブレーションは必ず複数シード平均で行うこと。**
- **改善3（検証セット補正）は、Optunaと組み合わせると危険**: 検証セットを3.3%変えただけで
  探索が別の領域（depth 4→8、L2が1/26）に着地した。少数の検証データ（535件）に対する
  Optuna探索は不安定で、検証セットの微修正がハイパーパラメータを通じて増幅される。
  検証セット補正を使うなら、**ハイパーパラメータは固定して**使う方が安全。